In [0]:
#%run ./transform_data ----- A décommmenter pour lancer les notebooks séparements

In [0]:
fact_unpivoted = (
    df_measures
    .select(
        "batch_id",
        "prd_line",
        F.lower(F.col("mesure_name")).alias("mesure_name"),
        F.col("mesure_value").alias("mesure_value"),
        F.col("date_debut"),
        F.col("date_fin")
    )
).cache()

calcul des outliers sur la base de la règle moyenne + 2 écarts types

In [0]:
# Optimisé : un seul groupBy au lieu de 2 + join
outliers_rules = fact_unpivoted.groupBy("mesure_name", "prd_line").agg(
    F.mean("mesure_value").alias("mean_value"),
    F.stddev("mesure_value").alias("std_value")
).withColumn(
    "outlier_upper_limit", F.col("mean_value") + (2 * F.col("std_value"))
).withColumn(
    "outlier_lower_limit", F.col("mean_value") - (2 * F.col("std_value"))
)

outliers_lower_limit = outliers_rules.withColumn(
    "outlier_lower_limit",
    F.when(F.col("outlier_lower_limit") < 0, 0).otherwise(F.col("outlier_lower_limit"))
)


calcul du top 20%

In [0]:
# jointure des métadonnées avec le DataFrame principal (broadcast sur dim table)
df_all = (
    fact_unpivoted.alias("a")
    .join(
        F.broadcast(df_metadata).alias("b"),
        (F.col("a.prd_line") == F.col("b.prd_line")) &
        (F.col("a.mesure_name") == F.col("b.measure_DE")),
        "inner"
    )
    .select(
        "a.batch_id",
        "a.prd_line",
        F.col("b.measure_DA").alias("mesure_name"),
        F.col("a.mesure_name").alias("mesure_name_DE"),
        "a.mesure_value",
        "a.date_debut",
        "a.date_fin",
        "b.categorie",
        "b.sequence",
        "b.label",
        "b.measure_type",
        "b.measure_description",
        F.col("b.columns").alias("columns_source"),
        "b.target_business"
    )
)
df_all.cache()

df_unpivoted_filtered = (
    df_all.alias("a")
    .join(
        F.broadcast(df_mesures_all_avec_cuves).alias("b"),
        F.col("a.batch_id") == F.col("b.batch_id"),
        "left"
    )
    .select(
        "a.*",
        "b.steep_vessel1",
        "b.steep_vessel2",
        "b.germ_vessel1",
        "b.germ_vessel2",
        "b.germ_vessel3",
        "b.germ_vessel4",
        "b.germ_vessel5",
        "b.germ_vessel6",
        "b.kiln_vessel1",
        "b.kiln_vessel2"
    )
)

df_unpivoted_filtered = (
    df_unpivoted_filtered
    .withColumn(
        "batch_prd_cell",
        F.when(
            (F.col("measure_type") == "BATCH_CYCLE_DURATION") & (F.col("categorie") == "TREMPE"),
            F.array("steep_vessel1", "steep_vessel2")
        )
        .when(
            (F.col("measure_type") == "BATCH_CYCLE_DURATION") & (F.col("categorie") == "GERMINATION"),
            F.array(
                "germ_vessel1",
                "germ_vessel2",
                "germ_vessel3",
                "germ_vessel4",
                "germ_vessel5",
                "germ_vessel6"
            )
        )
        .when(
            (F.col("measure_type") == "BATCH_CYCLE_DURATION") & (F.col("categorie") == "TOURAILLAGE"),
            F.array("kiln_vessel1", "kiln_vessel2")
        )
    )
    .withColumn(
        "batch_prd_cell",
        F.expr("filter(batch_prd_cell, x -> x IS NOT NULL)")
    )
)

df_with_outliers = df_unpivoted_filtered.alias("a").join(
    outliers_lower_limit.alias("b"),
    ((F.col("a.mesure_name") == F.col("b.mesure_name")) & 
    (F.col("a.prd_line") == F.col("b.prd_line"))),
    "left"
).select("a.*",
         "b.outlier_upper_limit",
         "b.outlier_lower_limit")

df_with_outliers_flg = df_with_outliers.withColumn("flg_outlier",
                                                   F.when((F.col("mesure_value") > F.col("outlier_upper_limit")) | (F.col("mesure_value") < F.col("outlier_lower_limit")), 1).otherwise(0))

In [0]:
table_tps_process_filtered = df_with_outliers_flg.alias("a").join(
    F.broadcast(dim_batches_specifications).alias("b"), 
    F.col("a.batch_id") == F.col("b.batch_id"),
    "left").select(
        "a.*",
        F.col("b.goods_weight").cast('float'),
        "b.requirement_specifications",
        "b.specy_name"
        
    )

table_tps_process_filtered = table_tps_process_filtered.alias("a").join(
    F.broadcast(batches_info).alias("b"),
    F.col("a.batch_id") == F.col("b.batch_id"),
    "left"
).select(
    "a.*",
    "b.id_plant_production_line"
)

In [0]:
# classer les batchs par groupe de tonnage (arrondie à 5 tonnes)
df_with_delta = table_tps_process_filtered.withColumn(
    'goods_weight_groups',
    F.floor(F.col('goods_weight') / 5) - (0 // 5) + 1
)

In [0]:
#ignorer les lignes où mesure_value est nul ou à 0 pour éviter de biaiser les résultats 
table_tps_process_filtered_0 = df_with_delta.filter(
    (F.col("mesure_value") > 0) | 
    ((F.col("mesure_value") == 0) & (F.col("mesure_name") == "attente_debut_germination"))
)

In [0]:
# Définir la fenêtre pour chaque groupe
window_spec = Window.partitionBy("prd_line", "goods_weight_groups", "mesure_name").orderBy("mesure_value")

# Calculer le rang de chaque ligne au sein de son groupe, basé sur mesure_value
df_with_ranks_tonnage = table_tps_process_filtered_0.withColumn("percent_rank_goods_weight", F.percent_rank().over(window_spec))

# Calculer le top 20% comme étant les valeurs ayant un percent_rank inférieur ou égal à 0.20
df_with_top20_flag_tonnage = df_with_ranks_tonnage.withColumn("flg_top20_tonnage", F.when(F.col("percent_rank_goods_weight") <= 0.20, 1).otherwise(0))

# Mettre aussi en cache le résultat du join pour les futures utilisations
df_with_top20_flag_tonnage.cache()

In [0]:
#Rejoindre avec les lignes initiales pour réintégrer les 0 et nulls

df_with_flags_all = (
    df_with_delta.alias("a")
    .join(
        df_with_top20_flag_tonnage.alias("b"),
        ((F.col("a.batch_id") == F.col("b.batch_id")) & (F.col("a.mesure_name") == F.col("b.mesure_name"))),
        "left"
    )
    .select("a.*", "b.flg_top20_tonnage")
    .fillna({"flg_top20_tonnage": 0})
)


# Mettre aussi en cache le résultat du join pour les futures utilisations
df_with_flags_all.cache()

In [0]:
# Calculer le 80ème percentile des 'mesure_value' pour les lignes du top 20% dans chaque groupe
top20_percentile_tonnage = df_with_flags_all.filter(F.col("flg_top20_tonnage") == 1) \
                            .groupBy("prd_line", "goods_weight_groups", "mesure_name") \
                            .agg(F.expr("percentile_approx(mesure_value, 0.80)").alias("top20_80th_percentile_tonnage"))

# Joindre ce 80ème percentile au DataFrame original
df_with_percentile_top20_tonnage = df_with_flags_all.join(top20_percentile_tonnage, 
                                          on=["prd_line", "goods_weight_groups", "mesure_name"], 
                                          how="left")

In [0]:
# Ajouter la colonne 'delta_percentile_top_20' qui est la différence entre 'mesure_value' et 'top20_percentile'
df_with_delta_tonnage = df_with_percentile_top20_tonnage.withColumn("delta_percentile_top_20_tonnage", F.col("mesure_value") - F.col("top20_80th_percentile_tonnage"))

# Mettre aussi en cache le résultat du join pour les futures utilisations
df_with_delta_tonnage.cache()

In [0]:
with_delta_target_business = df_with_delta_tonnage.withColumn("delta_target_business", F.col("mesure_value") - F.col("target_business"))

In [0]:
table_fact_batch_top_percentile = with_delta_target_business.select(
    "batch_id",
    "prd_line",
    "mesure_name",
    "mesure_value",
    "flg_top20_tonnage",
    "top20_80th_percentile_tonnage",
    "delta_percentile_top_20_tonnage",
    "categorie",
    "measure_type",
    "sequence",
    "label",
    "batch_prd_cell",
    "measure_description",
    "columns_source",
    "flg_outlier",
    "target_business",
    "delta_target_business",
    "id_plant_production_line"
)

table_fact_batch_top_percentile = table_fact_batch_top_percentile.dropDuplicates() 

on récupère maintenant la date de début et de fin

In [0]:
df_dates_utc = (
    table_fact_batch_top_percentile.alias("a")
    .join(
        df_all.select(
            "batch_id",
            "prd_line",
            "mesure_name",
            "date_debut",
            "date_fin"
        ).alias("b"),
        ["batch_id", "prd_line", "mesure_name"],
        "left"
    )
    .withColumn("date_debut", F.from_utc_timestamp("date_debut", "Europe/Paris"))
    .withColumn("date_fin", F.from_utc_timestamp("date_fin", "Europe/Paris"))
)


In [0]:
mesure_value_h = df_dates_utc.withColumn(
    "mesure_value_h",
    (F.col("mesure_value") / 60).cast(FloatType())
).withColumn(
    "top20_80th_percentile_h",
    (F.col("top20_80th_percentile_tonnage") / 60).cast(FloatType())
).withColumn(
    "delta_percentile_top_20_h",
    (F.col("delta_percentile_top_20_tonnage") / 60).cast(FloatType())
).withColumn(
    "target_business_h",
    (F.col("target_business") / 60).cast(FloatType())
).withColumn(
    "delta_target_business_h",
    (F.col("delta_target_business") / 60).cast(FloatType())
)

In [0]:
date_ref = mesure_value_h.alias("a").join(
    F.broadcast(dim_batches_specifications).alias("b"),
    F.col("a.batch_id") == F.col("b.batch_id"),
    "left"
).select("a.*",
         "b.date_fin_prod")


# On définit une colonne "date_ref" qui prend date_fin si dispo, sinon date_fin_prd
mesure_value_h_date_ref = date_ref.withColumn(
    "date_ref",
    F.coalesce(F.col("date_fin"), F.col("date_fin_prod"))
)

# Construction de df avec week, year et month_label basés sur date_ref
table_process_time_analyses = (
    mesure_value_h_date_ref
    .withColumn("week_mesure", F.weekofyear("date_ref"))
    .withColumn("year_mesure", F.year("date_ref"))
    .withColumn("month_num_mesure", F.month("date_ref"))
    .withColumn(
        "month_map_mesure",
        F.create_map([F.lit(x) for x in sum(mois_fr.items(), ())])
    )
    .withColumn(
    "month_name_mesure",
    F.col("month_map_mesure")[F.col("month_num_mesure")]
    )
    .drop("month_map_mesure", "date_ref", "date_fin_prod")
)

In [0]:
# Optimisé : utilisation de date_format natif au lieu de 12 when()
table_process_time_analyses_eng = table_process_time_analyses.withColumn(
    "month_name_mesure",
    F.date_format(F.make_date(F.lit(2024), F.col("month_num_mesure"), F.lit(1)), "MMMM")
)

In [0]:
table_process_time_analyses = table_process_time_analyses_eng \
    .withColumn("sequence", F.col("sequence").cast("integer"))

In [0]:
table_process_time_analyses = table_process_time_analyses.filter(F.col("columns_source").isNotNull())

In [0]:
table_process_time_analyses_with_date = table_process_time_analyses.withColumn(
    "start_date",
    to_date("date_debut")
)

In [0]:
fact_process_time_analyses = table_process_time_analyses_with_date.select(
    "batch_id",
    "mesure_name",
    "mesure_value",
    "flg_top20_tonnage",
    "top20_80th_percentile_tonnage",
    "delta_percentile_top_20_tonnage",
    "delta_target_business",
    "categorie",
    "measure_type",
    "sequence",
    "label",
    "batch_prd_cell",
    "measure_description",
    "columns_source",
    "start_date",
    "date_debut",
    "date_fin",
    "mesure_value_h",
    "top20_80th_percentile_h",
    "delta_percentile_top_20_h",
    "delta_target_business_h",
    "week_mesure",
    "year_mesure",
    "month_num_mesure",
    "month_name_mesure",
    "flg_outlier",
    "target_business",
    "target_business_h",
    "id_plant_production_line"
)

In [0]:
current_process= "fact_process_time_analyses"

In [0]:
target_table_process_time_analyses = current_catalog +"."+current_schema+"."+current_process
print(target_table_process_time_analyses)

In [0]:
all_columns =  fact_process_time_analyses.columns
#display(all_columns)

In [0]:
fact_process_time_analyses = fact_process_time_analyses.dropDuplicates(["mesure_name", "batch_id"])

In [0]:

# define the primary key 
primary_key = [    
    'mesure_name'
    ,'batch_id']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
fact_process_time_analyses_to_write = fact_process_time_analyses.withColumn(
    "columns_source",
    F.to_json(F.col("columns_source"))
)


In [0]:
handle_table_update(
    fact_process_time_analyses_to_write, 
    target_table_process_time_analyses, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )